# Setting up the environment

## Importing libraries

In [ ]:
import requests
import pandas as pd
import pprint
import time

## Checking for GPU runtime (if enabled)

In [ ]:
!nvidia-smi

Thu Jun 18 05:13:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Data Acquisition

## Visualizing data

We send a request to the TMDB API and fetch data. We visualise how the data looks.

In [ ]:
api_key = "8265bd1679663a7ea12ac168da84d2e8"
url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={api_key}&language=en-US&page=471"
response = requests.get(url)
data = response.json()
pprint.pprint(data)

In [ ]:
# data is a dictionary having the following keys
print(data.keys())

dict_keys(['page', 'results', 'total_pages', 'total_results'])


In [ ]:
# data['results'] is a list containing the results of all the movies in a page
data['results'][0]

{'adult': False,
 'backdrop_path': '/zIp8rPr0UzL2QhB5JCU04DqPeHQ.jpg',
 'genre_ids': [35, 80],
 'id': 12118,
 'title': 'Police Academy 3: Back in Training',
 'original_language': 'en',
 'original_title': 'Police Academy 3: Back in Training',
 'overview': "When police funding is cut, the Governor announces he must close one of the academies. To make it fair, the two police academies must compete against each other to stay in operation. Mauser persuades two officers in Lassard's academy to better his odds, but things don't quite turn out as expected...",
 'popularity': 3.7304,
 'poster_path': '/pBxGgWSR0CMaCVMA2kQS5MWU1z3.jpg',
 'release_date': '1986-03-20',
 'softcore': False,
 'video': False,
 'vote_average': 5.811,
 'vote_count': 1371}

In [ ]:
overview = data['results'][0]['overview']
pprint.pprint(overview)

('When police funding is cut, the Governor announces he must close one of the '
 'academies. To make it fair, the two police academies must compete against '
 "each other to stay in operation. Mauser persuades two officers in Lassard's "
 "academy to better his odds, but things don't quite turn out as expected...")


In [ ]:
total_pages = data['total_pages']
total_pages

547

In [ ]:
# Movie data is only present up to page 500. The rest of the pages do not contain anything meaningful
url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={api_key}&language=en-US&page=500"

response = requests.get(url)
page_data = response.json()
pprint.pprint(page_data)

## Extracting movie data

We extract the id, title, overview and genre_ids for each movie

In [ ]:
# Function to extract title, overview, and genre_ids of movies from each page
def extract_from_page(num_page, movie_data):
    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={num_page}"
    response = requests.get(url)
    page_data = response.json()

    # If page does not contain results
    if 'results' not in page_data:
        print(f"No 'results' key found on page {num_page}. Skipping this page.")
        return

    # Iterating over the list of results
    for result in page_data['results']:
        try:
            # Fetching movie data
            id = result.get('id')
            overview = result.get('overview')
            title = result.get('title')
            genre_ids = result.get('genre_ids')

            movie_data.append({
                'id': id,
                'title': title,
                'overview': overview,
                'genre_ids': genre_ids
            })
        except Exception as e:
            print(f"Error processing movie on page {num_page}: {e}")

In [ ]:
movie_data = [] # Initialize an empty list to store dictionaries of movie data

# Fetching movie data from all pages and storing them in a list
for page in range(1, total_pages + 1):
    extract_from_page(page, movie_data)
    time.sleep(0.1)

    if page % 10 == 0:
        print(f"Extracted data till page {page}")

# After the loop, create the DataFrame from the collected list
df = pd.DataFrame(movie_data)

print()
print(f"Total number of movies extracted: {len(df)}")
display(df.head())

Extracted data till page 10...
Extracted data till page 20...
Extracted data till page 30...
Extracted data till page 40...
Extracted data till page 50...
Extracted data till page 60...
Extracted data till page 70...
Extracted data till page 80...
Extracted data till page 90...
Extracted data till page 100...
Extracted data till page 110...
Extracted data till page 120...
Extracted data till page 130...
Extracted data till page 140...
Extracted data till page 150...
Extracted data till page 160...
Extracted data till page 170...
Extracted data till page 180...
Extracted data till page 190...
Extracted data till page 200...
Extracted data till page 210...
Extracted data till page 220...
Extracted data till page 230...
Extracted data till page 240...
Extracted data till page 250...
Extracted data till page 260...
Extracted data till page 270...
Extracted data till page 280...
Extracted data till page 290...
Extracted data till page 300...
Extracted data till page 310...
Extracted data ti

,id,title,overview,genre_ids
0,1007757,Swapped,"A small woodland creature and a majestic bird,...","[12, 16, 10751, 14]"
1,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"[18, 80]"
2,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","[18, 80]"
3,687163,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"[878, 12]"
4,936075,Michael,"The story of Michael Jackson, one of the most ...","[10402, 18]"


## Extracting genre data

We extract which genre-id corresponds to which genre and prepare a genre map

In [ ]:
url = "https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US"
response = requests.get(url)
genres = response.json()
pprint.pprint(genres)

{'genres': [{'id': 28, 'name': 'Action'},
            {'id': 12, 'name': 'Adventure'},
            {'id': 16, 'name': 'Animation'},
            {'id': 35, 'name': 'Comedy'},
            {'id': 80, 'name': 'Crime'},
            {'id': 99, 'name': 'Documentary'},
            {'id': 18, 'name': 'Drama'},
            {'id': 10751, 'name': 'Family'},
            {'id': 14, 'name': 'Fantasy'},
            {'id': 36, 'name': 'History'},
            {'id': 27, 'name': 'Horror'},
            {'id': 10402, 'name': 'Music'},
            {'id': 9648, 'name': 'Mystery'},
            {'id': 10749, 'name': 'Romance'},
            {'id': 878, 'name': 'Science Fiction'},
            {'id': 10770, 'name': 'TV Movie'},
            {'id': 53, 'name': 'Thriller'},
            {'id': 10752, 'name': 'War'},
            {'id': 37, 'name': 'Western'}]}


In [ ]:
genre_list = genres['genres']
genre_map = {genre['id']: genre['name'] for genre in genre_list}
genre_map

{28: 'Action',
 12: 'Adventure',
 16: 'Animation',
 35: 'Comedy',
 80: 'Crime',
 99: 'Documentary',
 18: 'Drama',
 10751: 'Family',
 14: 'Fantasy',
 36: 'History',
 27: 'Horror',
 10402: 'Music',
 9648: 'Mystery',
 10749: 'Romance',
 878: 'Science Fiction',
 10770: 'TV Movie',
 53: 'Thriller',
 10752: 'War',
 37: 'Western'}

## Extracting keywords

We extract the keywords associated with each movie.
NOTE: Be careful! Extracting the keywords for all the movies takes a lot of time. All the keywords have already been extracted and saved in `movies.json` file

In [ ]:
# Visualizing the data
url = f"https://api.themoviedb.org/3/movie/479040/keywords"
params = {"api_key": api_key}
response = requests.get(url, params=params)
keyword_data = response.json()
pprint.pprint(keyword_data)

{'id': 479040,
 'keywords': [{'id': 447, 'name': 'post-traumatic stress disorder (ptsd)'},
              {'id': 6149, 'name': 'police'},
              {'id': 1930, 'name': 'kidnapping'},
              {'id': 2357, 'name': 'cleveland'},
              {'id': 6019, 'name': 'human trafficking'},
              {'id': 6593, 'name': 'stripper'},
              {'id': 9826, 'name': 'murder'},
              {'id': 235261, 'name': 'abduction'}]}


In [ ]:
# Function to extract keywords for a given movie id
def get_keywords(movie_id):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    params = {"api_key": api_key}
    response = requests.get(url, params=params)

    if response.status_code != 200:
        return []

    data = response.json()
    keywords = [item['name'].replace(" ", "-") for item in data['keywords']]

    return keywords

In [ ]:
keyword_list = []
count = 0

# Extracting keywords for all the movies in the data frame
for id in df['id']:
    keywords = get_keywords(id)
    keyword_list.append(keywords)
    time.sleep(0.1)
    count += 1

    if count%10==0:
        print(f"Extracted keywords for {count} movies")

df['keywords'] = keyword_list
display(df.head())

## Saving the DataFrame to a JSON file

Use the `.to_json()` method to save the DataFrame to a JSON file. The `orient='records'` parameter ensures that each row is represented as a JSON object, which is a common and easy-to-read format.

In [ ]:
# Movie data
movie_file = 'movies.json'
df.to_json(movie_file, orient='records', indent=4)
print(f'DataFrame successfully saved to {movie_file}')

# Genre map
import json
genre_file = 'genres.json'
with open(genre_file, 'w') as f:
    json.dump(genre_map, f, indent=4)
print(f'DataFrame successfully saved to {genre_file}')

DataFrame successfully saved to movies.json


TypeError: dump() missing 1 required positional argument: 'fp'